In [0]:
from pyspark.sql.functions import col, count, avg, round

# Read from Silver
df = spark.table("workspace.default.silver_accidents")

# Aggregate by State + Severity
gold_df = df.groupBy("State", "Severity", "Sunrise_Sunset", "Weather_Condition") \
            .agg(
                count("*").alias("accident_count"),
                round(avg("Temperature_F"), 2).alias("avg_temp"),
                round(avg("Visibility_mi"), 2).alias("avg_visibility"),
                round(avg("Wind_Speed_mph"), 2).alias("avg_wind"),
                round(avg("Distance_mi"), 2).alias("avg_distance")
            )

# Save as Gold Delta Table
gold_df.write.format("delta").mode("overwrite").saveAsTable("workspace.default.gold_accidents")
print(f"Gold layer done! Rows: {gold_df.count()}")
gold_df.show(5)

In [0]:
from pyspark.sql.functions import col, count, sum, avg, round, when

# We group by hour, day_of_week , junction and traffic signal
infra_risk_df = df.groupBy("hour", "day_of_week", "Junction", "Traffic_Signal") \
    .agg(
        count("*").alias("total_accidents"),
        # Calculate % of high severity accidents 3 or 4
        round((sum(when(col("Severity") >= 3, 1).otherwise(0)) / count("*")) * 100, 2).alias("high_severity_pct"),
        # Average distance impacted - indicates how disruptive these accidents are
        round(avg("Distance_mi"), 2).alias("avg_impact_distance"),
        # humidity often leads to foggy conditions
        round(avg("Humidity_pct"), 2).alias("avg_humidity")
    ) \
    .orderBy("high_severity_pct", ascending=False)

infra_risk_df.write.format("delta").mode("overwrite").saveAsTable("workspace.default.gold_infra_risk")

print(f"Risk Summary Gold layer done! Rows: {infra_risk_df.count()}")
infra_risk_df.show(5)